# 01 · Carga de datos y diccionario de datos
**Proyecto:** Estimación de edad y análisis morfológico de abulones  
**Alumno:** Jorge Alberto Rodríguez Enríquez · **Unidad II – Preparación de datos**  
**Fase CRISP-DM:** 2. Comprensión de los datos

**Objetivo de este notebook:** cargar el archivo original `abalone.data`, revisar su estructura y construir el **diccionario de datos** del proyecto.

> ⚠️ Regla de integridad: el archivo de `data/raw` **solo se lee, nunca se modifica**.

## 1. Importar bibliotecas
- `pandas` → manejar tablas (DataFrames).
- `numpy` → cálculos numéricos.
- `Path` → manejar rutas de carpetas de forma segura en Windows y Linux.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Rutas del proyecto (el notebook vive en /notebooks, por eso subimos un nivel con "..")
RUTA_RAW = Path("..") / "data" / "raw" / "abalone.data"
RUTA_DOCS = Path("..") / "docs"

print("¿Existe el archivo original?", RUTA_RAW.exists())

¿Existe el archivo original? True


## 2. Cargar el dataset
El archivo **no trae encabezados**, por eso usamos `header=None` y le damos los nombres con `names=`.  
Los nombres salen de la ficha oficial de UCI y del archivo `abalone.names`.

In [2]:
COLUMNAS = ["Sex", "Length", "Diameter", "Height", "Whole_weight",
            "Shucked_weight", "Viscera_weight", "Shell_weight", "Rings"]

df = pd.read_csv(RUTA_RAW, header=None, names=COLUMNAS)
df.head()

,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


## 3. Tamaño y tipos de datos
`shape` devuelve (filas, columnas). `dtypes` indica cómo guardó pandas cada columna: `object` = texto, `float64` = decimal, `int64` = entero.

In [3]:
print(f"Registros: {df.shape[0]:,}")
print(f"Variables: {df.shape[1]}")
print()
print(df.dtypes)

Registros: 4,177
Variables: 9

Sex                   str
Length            float64
Diameter          float64
Height            float64
Whole_weight      float64
Shucked_weight    float64
Viscera_weight    float64
Shell_weight      float64
Rings               int64
dtype: object


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4177 entries, 0 to 4176
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Sex             4177 non-null   str    
 1   Length          4177 non-null   float64
 2   Diameter        4177 non-null   float64
 3   Height          4177 non-null   float64
 4   Whole_weight    4177 non-null   float64
 5   Shucked_weight  4177 non-null   float64
 6   Viscera_weight  4177 non-null   float64
 7   Shell_weight    4177 non-null   float64
 8   Rings           4177 non-null   int64  
dtypes: float64(7), int64(1), str(1)
memory usage: 293.8 KB


## 4. Revisión rápida de calidad
Contamos valores faltantes (nulos), filas duplicadas y valores únicos por columna.

In [5]:
print("Valores nulos por columna:")
print(df.isna().sum())
print()
print("Filas duplicadas:", df.duplicated().sum())
print()
print("Valores únicos por columna:")
print(df.nunique())

Valores nulos por columna:
Sex               0
Length            0
Diameter          0
Height            0
Whole_weight      0
Shucked_weight    0
Viscera_weight    0
Shell_weight      0
Rings             0
dtype: int64

Filas duplicadas: 0

Valores únicos por columna:
Sex                  3
Length             134
Diameter           111
Height              51
Whole_weight      2429
Shucked_weight    1515
Viscera_weight     880
Shell_weight       926
Rings               28
dtype: int64


## 5. Diccionario de datos
Un **diccionario de datos** describe cada variable: qué significa, su tipo, su unidad, su rol en el proyecto y su rango real observado.

📌 Los valores continuos del dataset están **escalados (divididos entre 200)**. Para obtener la medida real hay que **multiplicar por 200** (por ejemplo, 0.455 → 91 mm).

In [6]:
info = {
    "Sex":            ("Categórica nominal", "—",       "Predictora", "Sexo: M = macho, F = hembra, I = infante"),
    "Length":         ("Numérica continua",  "mm (/200)", "Predictora", "Medida más larga de la concha"),
    "Diameter":       ("Numérica continua",  "mm (/200)", "Predictora", "Medida perpendicular al largo"),
    "Height":         ("Numérica continua",  "mm (/200)", "Predictora", "Altura con la carne dentro de la concha"),
    "Whole_weight":   ("Numérica continua",  "g (/200)",  "Predictora", "Peso del abulón completo"),
    "Shucked_weight": ("Numérica continua",  "g (/200)",  "Predictora", "Peso de la carne"),
    "Viscera_weight": ("Numérica continua",  "g (/200)",  "Predictora", "Peso de las vísceras (después de desangrar)"),
    "Shell_weight":   ("Numérica continua",  "g (/200)",  "Predictora", "Peso de la concha seca"),
    "Rings":          ("Numérica entera",    "anillos",   "Objetivo",   "Número de anillos; edad ≈ Rings + 1.5 años"),
}

filas = []
for col in COLUMNAS:
    tipo, unidad, rol, desc = info[col]
    if df[col].dtype == "object":
        rango = ", ".join(sorted(df[col].unique()))
    else:
        rango = f"{df[col].min()} – {df[col].max()}"
    filas.append({
        "Variable": col, "Tipo": tipo, "Unidad": unidad, "Rol": rol,
        "Descripción": desc, "Rango observado": rango,
        "Nulos": int(df[col].isna().sum()), "Valores únicos": int(df[col].nunique()),
    })

diccionario = pd.DataFrame(filas)
diccionario

,Variable,Tipo,Unidad,Rol,Descripción,Rango observado,Nulos,Valores únicos
0,Sex,Categórica nominal,—,Predictora,"Sexo: M = macho, F = hembra, I = infante",F – M,0,3
1,Length,Numérica continua,mm (/200),Predictora,Medida más larga de la concha,0.075 – 0.815,0,134
2,Diameter,Numérica continua,mm (/200),Predictora,Medida perpendicular al largo,0.055 – 0.65,0,111
3,Height,Numérica continua,mm (/200),Predictora,Altura con la carne dentro de la concha,0.0 – 1.13,0,51
4,Whole_weight,Numérica continua,g (/200),Predictora,Peso del abulón completo,0.002 – 2.8255,0,2429
5,Shucked_weight,Numérica continua,g (/200),Predictora,Peso de la carne,0.001 – 1.488,0,1515
6,Viscera_weight,Numérica continua,g (/200),Predictora,Peso de las vísceras (después de desangrar),0.0005 – 0.76,0,880
7,Shell_weight,Numérica continua,g (/200),Predictora,Peso de la concha seca,0.0015 – 1.005,0,926
8,Rings,Numérica entera,anillos,Objetivo,Número de anillos; edad ≈ Rings + 1.5 años,1 – 29,0,28


## 6. Guardar el diccionario
Lo guardamos en `docs/` para incluirlo en el informe de la Unidad II.

In [7]:
RUTA_DOCS.mkdir(exist_ok=True)
diccionario.to_csv(RUTA_DOCS / "diccionario_datos.csv", index=False, encoding="utf-8-sig")
print("Diccionario guardado en docs/diccionario_datos.csv")

Diccionario guardado en docs/diccionario_datos.csv


## 7. Conclusiones
- El dataset tiene **4,177 registros y 9 variables**: 1 categórica (Sex), 7 numéricas continuas y 1 entera (Rings, la variable objetivo).
- **No hay valores nulos ni filas duplicadas.**
- Las medidas están escaladas entre 200; se tomará en cuenta al interpretar resultados.
- En el siguiente notebook (02) se explorarán distribuciones, relaciones y posibles valores atípicos.